# 02 - Modelling experiments

In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SUBMISSION = Path.cwd().resolve().parents[1]
PROJECT_ROOT = SUBMISSION.parent
DATA = PROJECT_ROOT / "data" / "chlorophyll_student_2015_2023.csv"

sys.path.insert(0, str(SUBMISSION / "best_model"))
sys.path.insert(0, str(SUBMISSION / "training_code"))
import utils

DATE, TARGET = utils.DATE, utils.TARGET

In [3]:
raw = pd.read_csv(DATA)
raw[DATE] = pd.to_datetime(raw[DATE])
raw = raw.sort_values(DATE).reset_index(drop=True)

raw.head(5)

,date,chlorophyll_a_mg_m3,sst_c,par_umol_m2_s,nitrate_umol_l,wind_speed_m_s,upwelling_index,mixed_layer_depth_m,salinity_psu,current_speed_m_s,river_discharge_index,cloud_fraction,surface_pressure_hpa,turbidity_ntu
0,2015-01-01,1.384479,11.481587,149.109963,6.015581,6.312241,0.888380,31.398514,35.267326,0.243345,1.454131,0.772424,1013.630651,2.712575
1,2015-01-02,1.163834,11.645005,102.709144,5.378356,6.419118,0.069684,32.582147,35.232201,0.305887,1.504132,0.720854,1013.746950,2.929409
2,2015-01-03,1.159372,11.306775,113.837502,6.347327,7.360378,1.600903,34.065341,35.221364,0.363735,1.290741,0.734412,1014.856950,2.631379
3,2015-01-04,1.370594,12.116982,173.914824,6.120668,7.730971,1.282715,34.538421,35.267070,0.402836,1.644759,0.642274,1013.131768,2.668686
4,2015-01-05,1.637080,11.822242,76.814006,5.257523,6.902393,0.519057,35.724733,35.329770,0.364641,1.234290,0.744293,1014.762930,2.520198


## 1. Protocol
1. First we will test a very basic model, and see how it performs, no lags, without seasonality, no clipping, no smearing, no log transform;
2. We test more lag_specs according to the EDA, we know the real variables that have signal;
3. OLS, Ridge or Lasso?
4. Does the annual cycle help?
5. Does making a non linear transformation on the target help making the R2 closer to 1?
6. Does clipping improve the R2, since it's a autoregressive model?
7. Does smearing in the opposite transform help?


In the file walk_forward.py, we defined a function that simply based on the configuration, and the simple raw dataframe, can make a cross-validation for the data between the years of 2015 and 2022, returns the average R2 score of the folds. The test years considered are from 2018 to 2022 (inclusive) and 2023 is left for an unbiased estimate to determine which model generalizes the best. 
The configuration is a dictionary with the following keys: lag_spec (dictionary), n_harmonics (int), log_target (bool) , clip_percentiles (tuple or None), smearing (bool), estimator (tuple of (estimator_name, params) 


So the protocol is simple: 
Define various configurations, loop trough them, call repeat(); Compare the average R2 across folds, and may the best win, and pass to the next stage. 



### Stage 1: lags

In [4]:
import walk_forward

test_years = [2018, 2019, 2020, 2021, 2022]
# For the first config we train an AR(1) model with OLS two important exogenous variables of the same day 
config_0 = {
    "lag_spec": {utils.TARGET: [1], "nitrate_umol_l": [0], "upwelling_index": [0]},
    "n_harmonics": 0,
    "log_target": False,
    "clip_percentiles": None,
    "smearing": False,
    "estimator": ("ridge", {"alpha": 1e-6}),
}

mean_r2, results = walk_forward.repeat(raw, config_0, test_years)
#Average persistence

def average_persistence(results, test_years): 
    persistences = [results[f'{year}']['baselines']['persistence']['r2'] for year in test_years]
    avg_persistence_r2 = np.sum(persistences) / len(test_years)
    return avg_persistence_r2 

def average_mean(results, test_years):
    means = [results[f'{year}']['baselines']['train_mean']['r2'] for year in test_years]
    avg_mean_r2 = np.sum(means) / len(test_years)
    return avg_mean_r2
    
print(f"Persistence: {average_persistence(results, test_years)}")  
print(f"Mean: {average_mean(results, test_years)}")  
print(f"Estimator: {mean_r2}")


Persistence: 0.6431305352613897
Mean: -0.015536537674832252
Estimator: 0.6994039570650388


### Conclusions: 
A simple estimator can have better results than the persistence and the mean (of course). Now our main objective is defining a bunch of configs and seeing which one gives the best result given a lag combination

#### Lag candidates

In [5]:
# Nitrate and upwelling act with a delay of 2-3 days,
# the other variables are enough with lags 0-1
# SST and PAR add nothing.
T = utils.TARGET

lag_candidates = {
    "ar1":                     {T: [1]},
    "ar2":                     {T: [1, 2]},
    "ar3":                     {T: [1, 2, 3]},
    "ar1 + nit/upw 0":         {T: [1], "nitrate_umol_l": [0], "upwelling_index": [0]},
    "ar1 + nit/upw 0-1":       {T: [1], "nitrate_umol_l": [0, 1], "upwelling_index": [0, 1]},
    "ar1 + nit/upw 0-2":       {T: [1], "nitrate_umol_l": [0, 1, 2], "upwelling_index": [0, 1, 2]},
    "ar1 + nit/upw 0-3":       {T: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3]},
    "ar1 + nit/upw 0-5":       {T: [1], "nitrate_umol_l": [0, 1, 2, 3, 4, 5], "upwelling_index": [0, 1, 2, 3, 4, 5]},
    "ar2 + nit/upw 0-3":       {T: [1, 2], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3]},
    "nit/upw 0-3 + wind/current/mld 0-1": {
        T: [1],
        "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1], "current_speed_m_s": [0, 1], "mixed_layer_depth_m": [0, 1],
    },
    "nit/upw 0-3 + all others 0-1": {
        T: [1],
        "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1], "current_speed_m_s": [0, 1], "mixed_layer_depth_m": [0, 1],
        "salinity_psu": [0, 1], "river_discharge_index": [0, 1], "cloud_fraction": [0, 1],
        "surface_pressure_hpa": [0, 1], "turbidity_ntu": [0, 1],
    },
    "nit/upw 0-3 + all others 0-1 + sst/par 0-1": {
        T: [1],
        "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1], "current_speed_m_s": [0, 1], "mixed_layer_depth_m": [0, 1],
        "salinity_psu": [0, 1], "river_discharge_index": [0, 1], "cloud_fraction": [0, 1],
        "surface_pressure_hpa": [0, 1], "turbidity_ntu": [0, 1],
        "sst_c": [0, 1], "par_umol_m2_s": [0, 1],
    },
}

results = list()

for name, lag_spec in lag_candidates.items(): 
    config_0["lag_spec"] = lag_spec 
    mean_r2, _ = walk_forward.repeat(raw, config_0, test_years) 
    pair = (f"{name}", mean_r2)
    results.append(pair)

results

[('ar1', np.float64(0.6695728524395231)),
 ('ar2', np.float64(0.6678648128302458)),
 ('ar3', np.float64(0.6673884211208458)),
 ('ar1 + nit/upw 0', np.float64(0.6994039570650388)),
 ('ar1 + nit/upw 0-1', np.float64(0.7262894167072819)),
 ('ar1 + nit/upw 0-2', np.float64(0.7600397832436184)),
 ('ar1 + nit/upw 0-3', np.float64(0.7735888165797213)),
 ('ar1 + nit/upw 0-5', np.float64(0.7734948862654274)),
 ('ar2 + nit/upw 0-3', np.float64(0.7732842789631331)),
 ('nit/upw 0-3 + wind/current/mld 0-1', np.float64(0.7788276460873463)),
 ('nit/upw 0-3 + all others 0-1', np.float64(0.7784091258166519)),
 ('nit/upw 0-3 + all others 0-1 + sst/par 0-1',
  np.float64(0.7786804954682631))]

### Conclusions:
1. More lags in the output make the model worse;
2. Nitrate/upwelling, in accordance to the EDA, after lag3 they do not give any gain. Actually, they generate a bit of noise -0.001;
3. The addition of the lags in the other variables besides the wind/current and layer make the performance worse -0.0004; Also the use of SST and PAR, increases by 0.0003, so it's considered noise.
4. Lets test configuration with everything else fixed, but making wind/current and mld bigger.


Best result:  ('nit/upw 0-3 + wind/current/mld 0-1', np.float64(0.7788276460873463))

#### Lag candidates for mld, wind and current, one at a time

In [6]:

lag_candidates_exo = {
    "mld 0-1":     {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "mixed_layer_depth_m": [0, 1]},
    "mld 0-2":     {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "mixed_layer_depth_m": [0, 1, 2]},
    "mld 0-3":     {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "mixed_layer_depth_m": [0, 1, 2, 3]},
    "mld 0-5":     {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "mixed_layer_depth_m": [0, 1, 2, 3, 4, 5]},
    "wind 0-1":    {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "wind_speed_m_s": [0, 1]},
    "wind 0-2":    {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "wind_speed_m_s": [0, 1, 2]},
    "wind 0-3":    {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "wind_speed_m_s": [0, 1, 2, 3]},
    "wind 0-5":    {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "wind_speed_m_s": [0, 1, 2, 3, 4, 5]},
    "current 0-1": {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "current_speed_m_s": [0, 1]},
    "current 0-2": {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "current_speed_m_s": [0, 1, 2]},
    "current 0-3": {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "current_speed_m_s": [0, 1, 2, 3]},
    "current 0-5": {utils.TARGET: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "current_speed_m_s": [0, 1, 2, 3, 4, 5]},
}

results = list()

for name, lag_spec in lag_candidates_exo.items(): 
    config_0["lag_spec"] = lag_spec 
    mean_r2, _ = walk_forward.repeat(raw, config_0, test_years) 
    pair = (f"{name}", mean_r2)
    results.append(pair)

results

[('mld 0-1', np.float64(0.7738333717208603)),
 ('mld 0-2', np.float64(0.7738996529780814)),
 ('mld 0-3', np.float64(0.7738946996876394)),
 ('mld 0-5', np.float64(0.7735737326949612)),
 ('wind 0-1', np.float64(0.7773218301525358)),
 ('wind 0-2', np.float64(0.7768711564787094)),
 ('wind 0-3', np.float64(0.7768296346585526)),
 ('wind 0-5', np.float64(0.7764665744934949)),
 ('current 0-1', np.float64(0.7741535903741003)),
 ('current 0-2', np.float64(0.7740918998956726)),
 ('current 0-3', np.float64(0.7731383467931691)),
 ('current 0-5', np.float64(0.7725900297583238))]

### Conclusions: 
1. Wind looks like the most important here; Let's see if this gain is actually true by comparing the gain between each fold;


In [7]:

wind_comparison_candidates = {
    "no wind":  {T: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3]},
    "wind 0-1": {T: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "wind_speed_m_s": [0, 1]},
"wind 0": {T: [1], "nitrate_umol_l": [0, 1, 2, 3], "upwelling_index": [0, 1, 2, 3], "wind_speed_m_s": [0]}}

results = list()

for name, lag_spec in wind_comparison_candidates.items(): 
    config_0["lag_spec"] = lag_spec 
    _, measurements = walk_forward.repeat(raw, config_0, test_years) 
    for i in range(len(test_years)):
        fold_r2 = measurements[f"{test_years[i]}"]["metrics"]["r2"]
        triplet = (f"{name}", i, fold_r2) 
        results.append(triplet)
results


[('no wind', 0, np.float64(0.7947816043123795)),
 ('no wind', 1, np.float64(0.7722986330528457)),
 ('no wind', 2, np.float64(0.6618541895592942)),
 ('no wind', 3, np.float64(0.8105520467810212)),
 ('no wind', 4, np.float64(0.828457609193066)),
 ('wind 0-1', 0, np.float64(0.8030445761569118)),
 ('wind 0-1', 1, np.float64(0.7780786677439859)),
 ('wind 0-1', 2, np.float64(0.6579189713959104)),
 ('wind 0-1', 3, np.float64(0.8166208209835933)),
 ('wind 0-1', 4, np.float64(0.8309461144822768)),
 ('wind 0', 0, np.float64(0.7954890896785456)),
 ('wind 0', 1, np.float64(0.7719850164300571)),
 ('wind 0', 2, np.float64(0.6614902893859388)),
 ('wind 0', 3, np.float64(0.81077849656682)),
 ('wind 0', 4, np.float64(0.8279447132314343))]

### Conclusion: 
The chosen lag spec is: output: 1; nitrate: 0 to 3; upwelling: 0 to 3; wind: 0 to 1 

### Stage 2: estimator and regularization

#### Estimator candidates: coarse search

In [8]:

config_stage2 = {
    "lag_spec": {
        utils.TARGET: [1],
        "nitrate_umol_l": [0, 1, 2, 3],
        "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1],
    },
    "n_harmonics": 0,
    "log_target": False,
    "clip_percentiles": None,
    "smearing": False,
    "estimator": ("ridge", {"alpha": 1e-6}),
}

estimator_candidates = {
    "ridge 1e-4":  ("ridge", {"alpha": 1e-4}),
    "ridge 1e-3":  ("ridge", {"alpha": 1e-3}),
    "ridge 1e-2":  ("ridge", {"alpha": 1e-2}),
    "ridge 1e-1":  ("ridge", {"alpha": 1e-1}),
    "ridge 1":     ("ridge", {"alpha": 1}),
    "ridge 10":    ("ridge", {"alpha": 10}),
    "ridge 100":   ("ridge", {"alpha": 100}),
    "ridge 1000":  ("ridge", {"alpha": 1000}),
    "lasso 1e-5":  ("lasso", {"alpha": 1e-5}),
    "lasso 1e-4":  ("lasso", {"alpha": 1e-4}),
    "lasso 1e-3":  ("lasso", {"alpha": 1e-3}),
    "lasso 1e-2":  ("lasso", {"alpha": 1e-2}),
    "lasso 1e-1":  ("lasso", {"alpha": 1e-1}),
    "lasso 1":     ("lasso", {"alpha": 1}),
} 


results = list()
base_r2, _ = walk_forward.repeat(raw, config_stage2, test_years)
results.append(base_r2)
for name, estimator_info in estimator_candidates.items(): 
    config_stage2["estimator"] =  estimator_info
    mean_r2, _ = walk_forward.repeat(raw, config_stage2, test_years) 
    pair = (f"{name}", mean_r2)
    results.append(pair)


results



[np.float64(0.7773218301525358),
 ('ridge 1e-4', np.float64(0.7773218292435125)),
 ('ridge 1e-3', np.float64(0.7773218209786086)),
 ('ridge 1e-2', np.float64(0.7773217382248653)),
 ('ridge 1e-1', np.float64(0.7773209002615893)),
 ('ridge 1', np.float64(0.7773115208855078)),
 ('ridge 10', np.float64(0.7771481424933011)),
 ('ridge 100', np.float64(0.7742639519579674)),
 ('ridge 1000', np.float64(0.741886928109881)),
 ('lasso 1e-5', np.float64(0.7773290597463739)),
 ('lasso 1e-4', np.float64(0.7773898177162256)),
 ('lasso 1e-3', np.float64(0.7776618434180195)),
 ('lasso 1e-2', np.float64(0.7771379790814933)),
 ('lasso 1e-1', np.float64(0.7252399512890879)),
 ('lasso 1', np.float64(-0.015853627331171037))]

### Conclusions: 
1. Ridge does not make the result better. Lasso has a maximum at 1e-3. There is no benefit now in using regularization. Lets retest at the end, after usi 

### Stage 3: seasonality, clipping, log and smearing

#### Candidates for each piece

In [9]:
config_stage3 = {
    "lag_spec": {
        utils.TARGET: [1],
        "nitrate_umol_l": [0, 1, 2, 3],
        "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1],
    },
    "n_harmonics": 0,
    "log_target": False,
    "clip_percentiles": None,
    "smearing": False,
    "estimator": ("lasso", {"alpha": 1e-3}),
}

harmonics_candidates = {
    "0 harmonics": {"n_harmonics": 0},
    "1 harmonic":  {"n_harmonics": 1},
    "2 harmonics": {"n_harmonics": 2},
    "3 harmonics": {"n_harmonics": 3},
     "4 harmonics": {"n_harmonics": 4},
}

results = list()

for name, harmonics in harmonics_candidates.items(): 
    config_stage3["n_harmonics"] =  harmonics["n_harmonics"]
    mean_r2, _ = walk_forward.repeat(raw, config_stage3, test_years) 
    pair = (f"{name}", mean_r2)
    results.append(pair)

results


[('0 harmonics', np.float64(0.7776618434180195)),
 ('1 harmonic', np.float64(0.779375293145829)),
 ('2 harmonics', np.float64(0.7891059346680328)),
 ('3 harmonics', np.float64(0.7871850832560774)),
 ('4 harmonics', np.float64(0.7863299950232546))]

### Best: ('2 harmonics', np.float64(0.7891059346680328))

In [10]:

config_stage4 = {
    "lag_spec": {
        utils.TARGET: [1],
        "nitrate_umol_l": [0, 1, 2, 3],
        "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1],
    },
    "n_harmonics": 2,
    "log_target": False,
    "clip_percentiles": None,
    "smearing": False,
    "estimator": ("lasso", {"alpha": 1e-3}),
}


log_candidates = {
    "no log": {"log_target": False},
    "log":    {"log_target": True},
}

#No log result is 0.7891059346680328 
#With log: 
config_stage4["log_target"] = True
mean_r2, _ = walk_forward.repeat(raw, config_stage4, test_years) 
mean_r2


np.float64(0.8038694890802182)

### Best: with log ((0.8038694890802182))

In [11]:
config_stage5 = {
    "lag_spec": {
        utils.TARGET: [1],
        "nitrate_umol_l": [0, 1, 2, 3],
        "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1],
    },
    "n_harmonics": 2,
    "log_target": True,
    "clip_percentiles": None,
    "smearing": False,
    "estimator": ("lasso", {"alpha": 1e-3}),
}



clip_candidates = {
    "no clip":    {"clip_percentiles": None},
    "clip 0.1":   {"clip_percentiles": (0.1, 99.9)},
    "clip 0.5":   {"clip_percentiles": (0.5, 99.5)},
    "clip 1":     {"clip_percentiles": (1, 99)},
    "clip 2.5":   {"clip_percentiles": (2.5, 97.5)},
}

results = list()

for name, clip_profile in clip_candidates.items(): 
    config_stage5["clip_percentiles"] = clip_profile["clip_percentiles"]
    mean_r2, _ = walk_forward.repeat(raw, config_stage5, test_years) 
    pair = (f"{name}", mean_r2)
    results.append(pair)

results

[('no clip', np.float64(0.8038694890802182)),
 ('clip 0.1', np.float64(0.8089853359044339)),
 ('clip 0.5', np.float64(0.8107795267949477)),
 ('clip 1', np.float64(0.8107823853414772)),
 ('clip 2.5', np.float64(0.8094857875908176))]

### Best: ('clip 0.5', np.float64(0.8107795267949477)),

In [12]:

config_stage6 = {
    "lag_spec": {
        utils.TARGET: [1],
        "nitrate_umol_l": [0, 1, 2, 3],
        "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1],
    },
    "n_harmonics": 2,
    "log_target": True,
    "clip_percentiles": (0.5, 99.5),
    "smearing": False,
    "estimator": ("lasso", {"alpha": 1e-3}),
}


smearing_candidates = {
    "no smearing": {"smearing": False},
    "smearing":    {"smearing": True},
}

#With no smearing:  ('clip 0.5', np.float64(0.8107795267949477)),

#With smearing: 
config_stage6["smearing"] = True
mean_r2, _ = walk_forward.repeat(raw, config_stage6, test_years) 
mean_r2


np.float64(0.8098109490041867)

#### Alpha candidates on the full configuration

In [13]:
config_stage6 = {
    "lag_spec": {
        utils.TARGET: [1],
        "nitrate_umol_l": [0, 1, 2, 3],
        "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1],
    },
    "n_harmonics": 2,
    "log_target": True,
    "clip_percentiles": (0.5, 99.5),
    "smearing": False,
    "estimator": ("lasso", {"alpha": 1e-3}),
}



alpha_candidates = {
    "lasso 1e-4": ("lasso", {"alpha": 1e-4}),
    "lasso 3e-4": ("lasso", {"alpha": 3e-4}),
    "lasso 5e-4": ("lasso", {"alpha": 5e-4}),
    "lasso 1e-3": ("lasso", {"alpha": 1e-3}),
    "lasso 2e-3": ("lasso", {"alpha": 2e-3}),
    "lasso 3e-3": ("lasso", {"alpha": 3e-3}),
    "lasso 1e-2": ("lasso", {"alpha": 1e-2}),
}


results = list()
base_r2, _ = walk_forward.repeat(raw, config_stage6, test_years)
results.append(base_r2)
for name, estimator_info in estimator_candidates.items(): 
    config_stage6["estimator"] =  estimator_info
    mean_r2, _ = walk_forward.repeat(raw, config_stage6, test_years) 
    pair = (f"{name}", mean_r2)
    results.append(pair)


results


[np.float64(0.8107795267949477),
 ('ridge 1e-4', np.float64(0.8113906237926484)),
 ('ridge 1e-3', np.float64(0.8113906220233025)),
 ('ridge 1e-2', np.float64(0.8113906042562288)),
 ('ridge 1e-1', np.float64(0.811390419255215)),
 ('ridge 1', np.float64(0.8113878662241849)),
 ('ridge 10', np.float64(0.811313152263995)),
 ('ridge 100', np.float64(0.8094248416299992)),
 ('ridge 1000', np.float64(0.7751481988800244)),
 ('lasso 1e-5', np.float64(0.8113904473517358)),
 ('lasso 1e-4', np.float64(0.8113954730606363)),
 ('lasso 1e-3', np.float64(0.8107795267949477)),
 ('lasso 1e-2', np.float64(0.8051009549936886)),
 ('lasso 1e-1', np.float64(0.6453426179295224)),
 ('lasso 1', np.float64(-0.011296875376743287))]

### Conclusion: 
Since the gain is not very big compared to noise, let's choose the simplest model: Lasso with 1e-3. With this alpha, LASSO discards the lags 0 and 1 of nitrate and upwelling, and only keeps lags 2 and 3, which leaves the same conclusion from EDA. 

### Selected configuration

In [14]:
final_config = {
    "lag_spec": {
        utils.TARGET: [1],
        "nitrate_umol_l": [0, 1, 2, 3],
        "upwelling_index": [0, 1, 2, 3],
        "wind_speed_m_s": [0, 1],
    },
    "n_harmonics": 2,
    "log_target": True,
    "clip_percentiles": (0.5, 99.5),
    "smearing": False,
    "estimator": ("lasso", {"alpha": 1e-3}),
}

r2, results = walk_forward.repeat(raw, final_config, [2023]) 
r2

np.float64(0.570338645477722)